In [2]:
# ==========================================
# 1. COLAB SETUP: DRIVE + REPO CODE
# ==========================================
import os
import pathlib
import subprocess
import sys

from google.colab import drive
from google.colab import userdata

REPO_URL = "https://github.com/DrHBSB/RadLE_CRASH_Lab.git"
REPO_REF = "codex/morning-meta-muse-spark-append"
REPO_DIR = pathlib.Path("/content/RadLE_CRASH_Lab")
SRC_DIR = REPO_DIR / "src"
MODULE_PATH = SRC_DIR / "radle_benchmark.py"
github_token = None

# Drive remains the home for images and benchmark outputs.
drive.mount("/content/drive")


def run_git(args, check=True):
    """Run git and print useful stderr without leaking the optional token."""
    result = subprocess.run(args, text=True, capture_output=True)
    stdout = result.stdout.replace(github_token, "***") if github_token else result.stdout
    stderr = result.stderr.replace(github_token, "***") if github_token else result.stderr
    if stdout.strip():
        print(stdout)
    if stderr.strip():
        print(stderr)
    if check and result.returncode != 0:
        hint = ""
        if "403" in stderr or "Write access to repository not granted" in stderr:
            hint = (
                " The GITHUB_TOKEN secret exists, but GitHub rejected it. "
                "Create a token that has read access to this private repository "
                "and permission to read repository contents."
            )
        raise RuntimeError(f"Git command failed with exit code {result.returncode}: {args[:2]}.{hint}")
    return result


# Colab does not automatically check out the GitHub branch used to open this notebook.
# Pin the code checkout to the branch that contains this notebook's model registry entries.
# For this private repo, add a Colab secret named GITHUB_TOKEN only if unauthenticated clone fails.
if (REPO_DIR / ".git").exists():
    run_git(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_REF])
    checkout_result = run_git(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=False)
    if checkout_result.returncode != 0:
        run_git(["git", "-C", str(REPO_DIR), "checkout", "-B", REPO_REF, f"origin/{REPO_REF}"])
    pull_result = run_git(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_REF], check=False)
    if pull_result.returncode != 0:
        raise RuntimeError(f"git pull failed for {REPO_REF}; restart the runtime or remove {REPO_DIR}, then rerun setup.")
elif not MODULE_PATH.exists():
    if REPO_DIR.exists() and any(REPO_DIR.iterdir()):
        raise RuntimeError(
            f"{REPO_DIR} exists but is not a Git checkout and {MODULE_PATH} was not found. "
            "Restart the Colab runtime or remove that folder, then rerun this setup cell."
        )

    try:
        github_token = userdata.get("GITHUB_TOKEN")
    except Exception:
        github_token = None

    if not github_token:
        raise RuntimeError(
            "This private GitHub repo needs a Colab secret named GITHUB_TOKEN "
            "so the Colab runtime can clone src/radle_benchmark.py. "
            "Add the secret, restart or rerun this setup cell, then continue."
        )

    clone_url = REPO_URL.replace("https://", f"https://x-access-token:{github_token}@")
    run_git(["git", "clone", "--branch", REPO_REF, clone_url, str(REPO_DIR)])

if not MODULE_PATH.exists():
    raise RuntimeError(f"Benchmark module not found after setup: {MODULE_PATH}")

current_ref = run_git(["git", "-C", str(REPO_DIR), "rev-parse", "--abbrev-ref", "HEAD"]).stdout.strip()
current_commit = run_git(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"]).stdout.strip()
print("Repo checkout:", current_ref, current_commit)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Ensure the Anthropic SDK is available.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic", "google-genai"], check=True)


Mounted at /content/drive
Cloning into '/content/RadLE_CRASH_Lab'...

codex/morning-meta-muse-spark-append

8029ad4

Repo checkout: codex/morning-meta-muse-spark-append 8029ad4


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'anthropic', 'google-genai'], returncode=0)

In [4]:
# ==========================================
# 2. API CLIENT + BENCHMARK IMPORTS
# ==========================================
import importlib
import inspect

import anthropic
import google.genai as google_genai
from openai import OpenAI

import radle_benchmark
import radle_meta_model_api_runtime

radle_benchmark = importlib.reload(radle_benchmark)
radle_meta_model_api_runtime = importlib.reload(radle_meta_model_api_runtime)
create_scorer_view = radle_benchmark.create_scorer_view
run_benchmark = radle_benchmark.run_benchmark
audit_benchmark_output = radle_benchmark.audit_benchmark_output
run_targeted_repair = radle_benchmark.run_targeted_repair

for _required_param in ("anthropic_client", "gemini_client", "meta_client", "backup_dir"):
    if _required_param not in inspect.signature(run_benchmark).parameters:
        raise RuntimeError(
            f"Loaded stale radle_benchmark missing {_required_param}. "
            "Rerun the setup/import cells after git pull, or restart the runtime."
        )

for _required_function in (
    "audit_benchmark_output",
    "run_targeted_repair",
    "build_run_paths",
    "promote_final_results",
    "export_public_release_tables",
):
    if not hasattr(radle_benchmark, _required_function):
        raise RuntimeError(
            f"Loaded stale radle_benchmark missing {_required_function}. "
            "Rerun the setup/import cells after git pull, or restart the runtime."
        )

if not hasattr(radle_meta_model_api_runtime, "make_openai_client"):
    raise RuntimeError(
        "Loaded Meta helper is missing make_openai_client. "
        "Rerun the setup/import cells after git pull, or restart the runtime."
    )

print(f"Loaded benchmark module: {radle_benchmark.__file__}")

try:
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    print("ERROR: Could not find OPENAI_API_KEY in Colab Secrets.")

try:
    os.environ["TEST_OPENROUTER_API_KEY"] = userdata.get("TEST_OPENROUTER_API_KEY")
except Exception:
    print("ERROR: Could not find TEST_OPENROUTER_API_KEY in Colab Secrets.")

openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

try:
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    print("ERROR: Could not find ANTHROPIC_API_KEY in Colab Secrets.")

anthropic_client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

try:
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:
    print("ERROR: Could not find GEMINI_API_KEY in Colab Secrets.")

gemini_client = google_genai.Client(api_key=os.environ["GEMINI_API_KEY"])

openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["TEST_OPENROUTER_API_KEY"],
)

meta_client = radle_meta_model_api_runtime.make_openai_client()
META_MODEL = radle_meta_model_api_runtime.get_model_config()
print("Meta API base URL:", os.environ.get("META_MODEL_API_BASE_URL", radle_meta_model_api_runtime.DEFAULT_BASE_URL))
print("Meta model:", META_MODEL["id"])


Loaded benchmark module: /content/RadLE_CRASH_Lab/src/radle_benchmark.py
Meta API base URL: https://api.meta.ai/v1
Meta model: muse-spark-1.1


In [5]:
# ==========================================
# 3. DRIVE PATHS + RUN CONFIG
# ==========================================
from pathlib import Path

# Full Grok 4.5 + GPT 5.6 Sol Pro + Muse Spark append for the canonical Morning run folder.
TEST_LIMIT = None
RUN_LABEL = "radle_v2"

dataset_root = Path("/content/drive/MyDrive/CRASH Lab/RaDLE/CONFIDENTIAL/RadLE v2 Dataset")
run_paths = radle_benchmark.build_run_paths(dataset_root, run_label=RUN_LABEL)

master_images_folder = run_paths["master_images_folder"]
final_output_csv = run_paths["raw_results_csv"]
raw_backup_dir = run_paths["raw_backup_dir"]
scorer_csv = run_paths["scorer_view_csv"]
repair_output_csv = run_paths["repair_results_csv"]
repair_call_log_csv = run_paths["repair_call_log_csv"]
repair_plan_csv = run_paths["repair_plan_csv"]
repair_backup_dir = run_paths["repair_backup_dir"]
final_results_csv = run_paths["final_results_csv"]
final_manifest_json = run_paths["final_manifest_json"]
public_release_dir = run_paths["public_release_dir"]

print("Run folder:", run_paths["run_root"])
print("Raw results CSV:", final_output_csv)

# Keep this focused so completed model columns are skipped and only these three models are appended.
EXPECTED_MODEL_NAMES = ["grok_4_5", "gpt_5_6_sol_pro", "muse_spark_1_1"]
DEBUG_MODEL_NAMES = ["grok_4_5", "gpt_5_6_sol_pro"]

if DEBUG_MODEL_NAMES is None:
    raise RuntimeError(
        "This combined append notebook refuses to run the full default model registry. "
        "Keep DEBUG_MODEL_NAMES = ['grok_4_5', 'gpt_5_6_sol_pro']."
    )

_available_model_names = {m["name"] for m in radle_benchmark.MODELS}
_unknown_model_names = sorted(set(DEBUG_MODEL_NAMES) - _available_model_names)
if _unknown_model_names:
    raise ValueError(f"Unknown model name(s): {_unknown_model_names}")

_model_by_name = {m["name"]: m for m in radle_benchmark.MODELS}
ACTIVE_MODELS = [_model_by_name[name] for name in DEBUG_MODEL_NAMES]
META_MODEL = radle_meta_model_api_runtime.get_model_config()
ALL_ACTIVE_MODELS = ACTIVE_MODELS + [META_MODEL]
_active_model_names = [m["name"] for m in ALL_ACTIVE_MODELS]
if _active_model_names != EXPECTED_MODEL_NAMES:
    raise RuntimeError(
        f"Wrong active models: {_active_model_names}. Expected exactly {EXPECTED_MODEL_NAMES}. "
        "Stop before running the benchmark cell."
    )

_active_model_by_name = {m["name"]: m for m in ACTIVE_MODELS}
_grok45_model = _active_model_by_name["grok_4_5"]
if _grok45_model.get("id") != "x-ai/grok-4.5":
    raise RuntimeError(f"Wrong Grok 4.5 model id: {_grok45_model.get('id')}")
if _grok45_model.get("provider_routing") != {"only": ["xAI"], "allow_fallbacks": False}:
    raise RuntimeError(f"Missing xAI provider routing: {_grok45_model.get('provider_routing')}")
_gpt56_model = _active_model_by_name["gpt_5_6_sol_pro"]
if _gpt56_model.get("id") != "openai/gpt-5.6-sol-pro":
    raise RuntimeError(f"Wrong GPT 5.6 Sol Pro model id: {_gpt56_model.get('id')}")
if _gpt56_model.get("extra") != {"reasoning": {"effort": "high"}}:
    raise RuntimeError(f"Wrong GPT 5.6 reasoning config: {_gpt56_model.get('extra')}")
if _gpt56_model.get("provider_routing") != {"only": ["OpenAI"], "allow_fallbacks": False}:
    raise RuntimeError(f"Missing OpenAI provider routing: {_gpt56_model.get('provider_routing')}")

if META_MODEL.get("id") != "muse-spark-1.1" or META_MODEL.get("provider") != "meta_model_api":
    raise RuntimeError(f"Wrong Meta model config: {META_MODEL}")

print("Running models:", _active_model_names)
print("Model IDs:", {m["name"]: m["id"] for m in ALL_ACTIVE_MODELS})
print("Provider routing:", {m["name"]: m.get("provider_routing") for m in ALL_ACTIVE_MODELS})
print("Provider types:", {m["name"]: m.get("provider", "openrouter") for m in ALL_ACTIVE_MODELS})


Run folder: /content/drive/MyDrive/CRASH Lab/RaDLE/CONFIDENTIAL/RadLE v2 Dataset/Runs/radle_v2
Raw results CSV: /content/drive/MyDrive/CRASH Lab/RaDLE/CONFIDENTIAL/RadLE v2 Dataset/Runs/radle_v2/raw/results.csv
Running models: ['grok_4_5', 'gpt_5_6_sol_pro', 'muse_spark_1_1']
Model IDs: {'grok_4_5': 'x-ai/grok-4.5', 'gpt_5_6_sol_pro': 'openai/gpt-5.6-sol-pro', 'muse_spark_1_1': 'muse-spark-1.1'}
Provider routing: {'grok_4_5': {'only': ['xAI'], 'allow_fallbacks': False}, 'gpt_5_6_sol_pro': {'only': ['OpenAI'], 'allow_fallbacks': False}, 'muse_spark_1_1': None}
Provider types: {'grok_4_5': 'openrouter', 'gpt_5_6_sol_pro': 'openrouter', 'muse_spark_1_1': 'meta_model_api'}


In [6]:
# ==========================================
# 4. RUN BENCHMARK
# ==========================================
# One serial writer owns the canonical CSV. Existing accepted Grok/GPT cells skip;
# Muse Spark uses meta_client and writes only its missing model cells.
df_final = run_benchmark(
    client=openrouter_client,
    openai_client=openai_client,
    anthropic_client=anthropic_client,
    gemini_client=gemini_client,
    meta_client=meta_client,
    image_folder=master_images_folder,
    output_csv=final_output_csv,
    test_limit=TEST_LIMIT,
    models=ALL_ACTIVE_MODELS,
    backup_dir=raw_backup_dir,
    resume=True,
)

print("\nFINAL DATAFRAME PREVIEW:")
from IPython.display import display

display(df_final.head())


Resuming existing benchmark output: /content/drive/MyDrive/CRASH Lab/RaDLE/CONFIDENTIAL/RadLE v2 Dataset/Runs/radle_v2/raw/results.csv | rows=200
Processing 200 unique cases across 3 models...

[1/200] Case ID: 1 (1 images)
  -> grok_4_5... SKIP (accepted_clean_diagnosis)
  -> gpt_5_6_sol_pro... SKIP (accepted_clean_diagnosis)
  -> muse_spark_1_1... SKIP (accepted_clean_diagnosis)
[2/200] Case ID: 2 (1 images)
  -> grok_4_5... SKIP (accepted_clean_diagnosis)
  -> gpt_5_6_sol_pro... SKIP (accepted_clean_diagnosis)
  -> muse_spark_1_1... SKIP (accepted_clean_diagnosis)
[3/200] Case ID: 3 (1 images)
  -> grok_4_5... SKIP (accepted_clean_diagnosis)
  -> gpt_5_6_sol_pro... SKIP (accepted_clean_diagnosis)
  -> muse_spark_1_1... SKIP (accepted_clean_diagnosis)
[4/200] Case ID: 4 (1 images)
  -> grok_4_5... SKIP (accepted_clean_diagnosis)
  -> gpt_5_6_sol_pro... SKIP (accepted_clean_diagnosis)
  -> muse_spark_1_1... SKIP (accepted_clean_diagnosis)
[5/200] Case ID: 5 (1 images)
  -> grok_4_5...

,Master_Case_ID,Associated_Images,Image_SHA256,Diagnosis_gpt_5_5,Likert_gpt_5_5,Prompt_Tokens_gpt_5_5,Total_Tokens_Out_gpt_5_5,Reasoning_Tokens_gpt_5_5,Latency_gpt_5_5,Provider_gpt_5_5,...,Provider_muse_spark_1_1,Timestamp_UTC_muse_spark_1_1,Reasoning_muse_spark_1_1,Reasoning_Raw_muse_spark_1_1,Reasoning_Details_muse_spark_1_1,Actual_Request_Extra_muse_spark_1_1,Grok_Fallback_Used_muse_spark_1_1,OpenRouter_Response_Model_muse_spark_1_1,Usage_JSON_muse_spark_1_1,Raw_Response_muse_spark_1_1
0,1,1.png,6c737472aac9d936,Carotid-cavernous fistula,4,1284,311,279,10.9,OpenAI,...,Meta Model API,2026-07-09T21:00:14.711204+00:00,NaN,NaN,NaN,NaN,NaN,muse-spark-1.1,"{""completion_tokens"": 675, ""prompt_tokens"": 13...","{""diagnosis"": ""Left carotid cavernous fistula""..."
1,2,2.png,8d62d1d2ae0e582b,Cerebral autosomal dominant arteriopathy with ...,3,995,557,512,17.2,OpenAI,...,Meta Model API,2026-07-09T21:00:24.386035+00:00,NaN,NaN,NaN,NaN,NaN,muse-spark-1.1,"{""completion_tokens"": 880, ""prompt_tokens"": 10...","{""diagnosis"": ""Central pontine myelinolysis"", ..."
2,3,3.png,eb8f8768c2407298,Dysplastic cerebellar gangliocytoma,4,1271,354,321,11.0,OpenAI,...,Meta Model API,2026-07-09T21:00:36.109419+00:00,NaN,NaN,NaN,NaN,NaN,muse-spark-1.1,"{""completion_tokens"": 1226, ""prompt_tokens"": 1...","{""diagnosis"": ""Lhermitte-Duclos disease"", ""lik..."
3,4,4.png,f72d9953cf3a6068,Anterior sacral meningocele,4,1243,449,420,16.8,OpenAI,...,Meta Model API,2026-07-09T21:00:53.251499+00:00,NaN,NaN,NaN,NaN,NaN,muse-spark-1.1,"{""completion_tokens"": 1943, ""prompt_tokens"": 1...","{""diagnosis"": ""Anterior sacral meningocele"", ""..."
4,5,5.png,9cfe77862e65dbbc,Baastrup disease,3,1326,2094,2048,65.4,OpenAI,...,Meta Model API,2026-07-09T21:01:33.212614+00:00,NaN,NaN,NaN,NaN,NaN,muse-spark-1.1,"{""completion_tokens"": 4818, ""prompt_tokens"": 1...","{""diagnosis"": ""Baastrup disease"", ""likert_scor..."


In [7]:
# ==========================================
# 5. SCORER'S VIEW & PIVOT
# ==========================================
df_scorer, display_df, scorer_csv = create_scorer_view(final_output_csv, scorer_csv=scorer_csv)

print(f"Scorer version saved to: {scorer_csv}")
print("\nTRANSPOSED CONSENSUS VIEW:")
display(display_df)


Scorer version saved to: /content/drive/MyDrive/CRASH Lab/RaDLE/CONFIDENTIAL/RadLE v2 Dataset/Runs/radle_v2/scorer/scorer_view.csv

TRANSPOSED CONSENSUS VIEW:


Master_Case_ID,1,2,3,4,5,6,7,8,9,10,...,191,192,193,194,195,196,197,198,199,200
Associated_Images,1.png,2.png,3.png,4.png,5.png,6.png,7.png,8.png,9.png,10.png,...,191.png,192.png,193.png,194.png,195.png,196.png,197.png,198.png,199.png,200.png
Diagnosis_gpt_5_5,Carotid-cavernous fistula,Cerebral autosomal dominant arteriopathy with ...,Dysplastic cerebellar gangliocytoma,Anterior sacral meningocele,Baastrup disease,Rathke cleft cyst,Bilateral retinoblastoma,Eagle syndrome,Calcaneal intraosseous lipoma,Calcaneonavicular coalition,...,Azygos lobe,Moyamoya disease,Tuberous sclerosis complex,Left slipped capital femoral epiphysis,Right-sided aortic arch,Splenic laceration,Rhizomelic chondrodysplasia punctata,Trichorhinophalangeal syndrome,Bilateral superior semicircular canal dehiscence,Low-flow venous malformation
Diagnosis_claude_4_8_opus,Carotid-cavernous fistula with dilated superio...,Brainstem (pontine) glioma,Hemangioblastoma,Myxopapillary ependymoma,Lumbar spondylolisthesis with spinal canal ste...,Rathke cleft cyst,Left orbital cavernous hemangioma,Eagle syndrome,Intraosseous lipoma of the calcaneus,Avulsion fracture of the navicular tuberosity,...,I don't know,Cerebral arteriovenous malformation,Tuberous sclerosis complex,Ovarian mature cystic teratoma (dermoid cyst),Thoracic aortic aneurysm,Hepatomegaly,Rickets,Normal hand radiograph,Middle ear cholesteatoma,Myxoid liposarcoma
Diagnosis_gemini_3_1_pro,Carotid-cavernous fistula,Multiple system atrophy,Lhermitte-Duclos disease,Lipomyelomeningocele,Tethered cord syndrome,Pituitary apoplexy,Orbital emphysema,Eagle syndrome,Intraosseous lipoma,Calcaneonavicular coalition,...,Situs inversus,Moyamoya disease,Tuberous sclerosis,Proximal focal femoral deficiency,Brachiocephalic artery aneurysm,Transient hepatic attenuation difference,Rhizomelic chondrodysplasia punctata,Scleroderma,Petrous apex cholesterol granuloma,Venous malformation
Diagnosis_grok_4_20,I don't know,Glioblastoma,I don't know,I don't know,I don't know,Chiari one malformation,I don't know,I don't know,Osteoid osteoma,I don't know,...,I don't know,Cerebral arteriovenous malformation,I don't know,I don't know,Pulmonary arteriovenous malformation,Hepatic hemangioma,Metaphyseal dysplasia,I don't know,I don't know,Myxoid liposarcoma
Diagnosis_qwen_3_7_plus,Carotid cavernous fistula,Multiple system atrophy,Lhermitte-Duclos disease,Anterior sacral meningocele,Kummell disease,Rathke's cleft cyst,Squamous cell carcinoma,Eagle syndrome,Intraosseous lipoma,Calcaneonavicular coalition,...,Cardiomegaly,Meningioma,Multiplesclerosis,Slipped capital femoral epiphysis,Ascending aortic aneurysm,Hepatocellular carcinoma,Rhizomelic chondrodysplasia punctata,Pseudohypoparathyroidism,Cholesterol granuloma,Varicose veins
Diagnosis_gemma_4_31b,Carotid-cavernous fistula,Ependymoma,Hemangioblastoma,Tailgut cyst,Tethered cord syndrome,Rathke cleft cyst,Mucormycosis,Fibrodysplasia ossificans progressiva,Aneurysmal bone cyst,Talar neck fracture,...,Situs inversus totalis,Moyamoya disease,Multiple Sclerosis,Avascular necrosis of the femoral head,Thymoma,Cirrhosis,Rhizomelic Chondrodysplasia Punctata,Albright hereditary osteodystrophy,Cholesterol granuloma of the petrous apex,Ganglion cyst
Diagnosis_llama_4_maverick,Mucocele,Pontine Glioma,Hemangioblastoma,Tethered Cord Syndrome,Lumbar disc herniation,Rathke cleft cyst,Cavernous hemangioma,Mandible fracture,Intraosseous Lipoma,Os trigonum syndrome,...,Left lower lobe collapse,Moyamoya disease,Multiple Sclerosis,Femoral neck fracture,Substernal Thyroid Goiter,Splenic laceration,Langerhans cell histiocytosis,Rheumatoid arthritis,Cochlear Dysplasia,Subcutaneous Schwannoma
Diagnosis_mistral_large_3_2512,Renal artery aneurysm,Acute ischemic stroke involving the posterior ...,Hepatocellular carcinoma with diffusion restri...,Lumbar spondylolisthesis with degenerative cha...,Lumbar spinal osteomyelitis with epidural abscess,Hepatic hemangioma,Maxillary sinus fungal ball (mycetoma),Mandibular amelobla

In [8]:
# ==========================================
# 6. READ-ONLY OUTPUT AUDIT
# ==========================================
# In append-smoke mode, the canonical raw CSV already contains prior 200-case rows.
# Audit a temporary subset so unrun rows are not reported as repair targets.
audit_raw_csv = final_output_csv
audit_expected_case_ids = None
if TEST_LIMIT is not None:
    import pandas as pd

    _expected_case_ids = [str(i) for i in range(1, TEST_LIMIT + 1)]
    _audit_df = pd.read_csv(final_output_csv, dtype={"Master_Case_ID": str})
    _audit_subset_df = _audit_df[_audit_df["Master_Case_ID"].astype(str).isin(_expected_case_ids)].copy()
    _audit_tmp_csv = Path("/tmp") / f"{RUN_LABEL}_smoke_audit_first_{TEST_LIMIT}_cases.csv"
    _audit_subset_df.to_csv(_audit_tmp_csv, index=False)
    audit_raw_csv = _audit_tmp_csv
    audit_expected_case_ids = range(1, TEST_LIMIT + 1)
    print("Smoke audit subset:", audit_raw_csv)

audit_results = audit_benchmark_output(
    raw_csv=audit_raw_csv,
    models=ALL_ACTIVE_MODELS,
    expected_case_ids=audit_expected_case_ids,
)

print("DATASET INTEGRITY:")
display(audit_results["dataset_integrity"])

print("\nBUCKET SUMMARY:")
display(audit_results["bucket_summary"])

print("\nSTATUS SUMMARY:")
display(audit_results["status_summary"])

print("\nNO-API CLEANUP TARGETS:")
display(audit_results["no_paid_cleanup"].head(100))

print("\nREPAIR TARGETS:")
display(audit_results["repair_targets"].head(100))

print("\nANALYSIS FLAGS")
display(audit_results["analysis_flags"].head(100))

print("\nPROVIDER CONTENT BLOCKS:")
display(audit_results["provider_content_blocks"].head(100))


DATASET INTEGRITY:


,metric,value
0,rows,200
1,unique_cases,200
2,columns,259
3,models_audited,3
4,expected_case_model_cells,600
5,duplicate_case_ids,none
6,missing_expected_case_ids,none
7,extra_case_ids,none



BUCKET SUMMARY:


,bucket,cells
0,accepted,600



STATUS SUMMARY:


...

,Master_Case_ID,Associated_Images,Image_SHA256,model,bucket,status,reason,needs_api_repair,repair_attempts_so_far,max_attempts,...,timestamp_utc,completion_tokens,prompt_tokens,reasoning_tokens,raw_len,hit_max_tokens,raw_preview,rescued_diag,rescued_likert,rescue_method



ANALYSIS FLAGS


,Master_Case_ID,Associated_Images,Image_SHA256,model,bucket,status,reason,needs_api_repair,repair_attempts_so_far,max_attempts,...,timestamp_utc,completion_tokens,prompt_tokens,reasoning_tokens,raw_len,hit_max_tokens,raw_preview,rescued_diag,rescued_likert,rescue_method



PROVIDER CONTENT BLOCKS:


,Master_Case_ID,Associated_Images,Image_SHA256,model,bucket,status,reason,needs_api_repair,repair_attempts_so_far,max_attempts,...,timestamp_utc,completion_tokens,prompt_tokens,reasoning_tokens,raw_len,hit_max_tokens,raw_preview,rescued_diag,rescued_likert,rescue_method


In [9]:
# ==========================================
# 7. TARGETED REPAIR PLAN / RUN
# ==========================================
# Keep REPAIR_CONFIRMATION="NO" to preview the repair plan without API calls or file writes.
# Change to "YES_REPAIR_10" for a capped repair test or "YES_REPAIR_ALL" for all eligible targets.
REPAIR_CONFIRMATION = "NO"

repair_results = run_targeted_repair(
    client=openrouter_client,
    openai_client=openai_client,
    anthropic_client=anthropic_client,
    gemini_client=gemini_client,
    meta_client=meta_client,
    image_folder=master_images_folder,
    input_csv=final_output_csv,
    output_csv=repair_output_csv,
    repair_call_log_csv=repair_call_log_csv,
    repair_plan_csv=repair_plan_csv,
    confirmation=REPAIR_CONFIRMATION,
    models=ALL_ACTIVE_MODELS,
    backup_dir=repair_backup_dir,
)

print("\nNO-API CLEANUP PLAN PREVIEW:")
display(repair_results["no_paid_cleanup_plan"].head(100))

print("\nREPAIR PLAN PREVIEW:")
display(repair_results["repair_plan"].head(100))
print("API calls this run:", repair_results["api_calls_this_run"])
print("No-API cleanups applied:", repair_results["no_paid_cleanups_applied"])
print("Repair input CSV:", final_output_csv)
print("Repair output CSV:", repair_results["output_csv"])
print("Repair call log CSV:", repair_results["repair_call_log_csv"])



=== TARGETED REPAIR PLAN ===
Input CSV: /content/drive/MyDrive/CRASH Lab/RaDLE/CONFIDENTIAL/RadLE v2 Dataset/Runs/radle_v2/raw/results.csv
Output CSV: /content/drive/MyDrive/CRASH Lab/RaDLE/CONFIDENTIAL/RadLE v2 Dataset/Runs/radle_v2/repair/repaired_results.csv
Repair call log: /content/drive/MyDrive/CRASH Lab/RaDLE/CONFIDENTIAL/RadLE v2 Dataset/Runs/radle_v2/repair/repair_call_log.csv
No-API cleanup rows / affected cells: 0
Repair plan rows / affected cells: 0
confirmation was NO. No API calls were made, no cleanups were applied, and no files were written.

NO-API CLEANUP PLAN PREVIEW:


""



REPAIR PLAN PREVIEW:


""


API calls this run: 0
No-API cleanups applied: 0
Repair input CSV: /content/drive/MyDrive/CRASH Lab/RaDLE/CONFIDENTIAL/RadLE v2 Dataset/Runs/radle_v2/raw/results.csv
Repair output CSV: /content/drive/MyDrive/CRASH Lab/RaDLE/CONFIDENTIAL/RadLE v2 Dataset/Runs/radle_v2/repair/repaired_results.csv
Repair call log CSV: /content/drive/MyDrive/CRASH Lab/RaDLE/CONFIDENTIAL/RadLE v2 Dataset/Runs/radle_v2/repair/repair_call_log.csv


In [10]:
# ==========================================
# 8. PROMOTE PRIVATE FINAL FILE
# ==========================================
if TEST_LIMIT is not None:
    raise RuntimeError(
        "Do not promote partial smoke output. Set TEST_LIMIT=None after the full append run, "
        "then rerun benchmark/audit/repair before promotion."
    )

repair_confirmed = globals().get("REPAIR_CONFIRMATION", "NO") != "NO"
repair_output_ready = repair_confirmed and Path(repair_output_csv).exists()
private_final_source_csv = repair_output_csv if repair_output_ready else final_output_csv
private_final_source_label = "repaired" if repair_output_ready else "raw"

final_manifest = radle_benchmark.promote_final_results(
    source_csv=private_final_source_csv,
    final_csv=final_results_csv,
    manifest_json=final_manifest_json,
    run_id=run_paths["run_id"],
    source_label=private_final_source_label,
    metadata={
        "run_label": RUN_LABEL,
        "test_limit": TEST_LIMIT if TEST_LIMIT is not None else "full",
    },
)

print("Private final source:", private_final_source_csv)
print("Private final CSV:", final_results_csv)
print("Private final manifest:", final_manifest_json)
print("Private final SHA256:", final_manifest["sha256"])


Private final source: /content/drive/MyDrive/CRASH Lab/RaDLE/CONFIDENTIAL/RadLE v2 Dataset/Runs/radle_v2/raw/results.csv
Private final CSV: /content/drive/MyDrive/CRASH Lab/RaDLE/CONFIDENTIAL/RadLE v2 Dataset/Runs/radle_v2/final/RadLE_v2_results_final.csv
Private final manifest: /content/drive/MyDrive/CRASH Lab/RaDLE/CONFIDENTIAL/RadLE v2 Dataset/Runs/radle_v2/final/RadLE_v2_results_final_manifest.json
Private final SHA256: 91b641c0953b6a81ff25b2bd962aea6ff8acb22e9d605976d4f2509cec276311


In [11]:
# ==========================================
# 9. EXPORT ANSWER-FREE PUBLIC RELEASE TABLES
# ==========================================
if TEST_LIMIT is not None:
    raise RuntimeError(
        "Do not export public release tables from partial smoke output. "
        "Set TEST_LIMIT=None after the full append run and final promotion."
    )

public_results_source_csv = final_results_csv if Path(final_results_csv).exists() else final_output_csv
public_call_log_csv = repair_call_log_csv if Path(repair_call_log_csv).exists() else None

public_release_files = radle_benchmark.export_public_release_tables(
    results_csv=public_results_source_csv,
    output_dir=public_release_dir,
    models=ALL_ACTIVE_MODELS,
    call_log_csv=public_call_log_csv,
    run_id=run_paths["run_id"],
)

print("Public release source CSV:", public_results_source_csv)
print("Public case-model CSV:", public_release_files["case_model_csv"])
print("Public model summary CSV:", public_release_files["summary_csv"])
print("Public sanitized call log CSV:", public_release_files["sanitized_call_log_csv"])
print("Public manifest:", public_release_files["manifest_json"])


Public release source CSV: /content/drive/MyDrive/CRASH Lab/RaDLE/CONFIDENTIAL/RadLE v2 Dataset/Runs/radle_v2/final/RadLE_v2_results_final.csv
Public case-model CSV: /content/drive/MyDrive/CRASH Lab/RaDLE/CONFIDENTIAL/RadLE v2 Dataset/Runs/radle_v2/public_release/RadLE_v2_public_model_results.csv
Public model summary CSV: /content/drive/MyDrive/CRASH Lab/RaDLE/CONFIDENTIAL/RadLE v2 Dataset/Runs/radle_v2/public_release/RadLE_v2_public_model_summary.csv
Public sanitized call log CSV: /content/drive/MyDrive/CRASH Lab/RaDLE/CONFIDENTIAL/RadLE v2 Dataset/Runs/radle_v2/public_release/RadLE_v2_public_sanitized_call_log.csv
Public manifest: /content/drive/MyDrive/CRASH Lab/RaDLE/CONFIDENTIAL/RadLE v2 Dataset/Runs/radle_v2/public_release/RadLE_v2_public_manifest.json
